# 06 - Bronze Layer Quality Check

Observability notebook for the Bronze layer. Validates schema enforcement, dtype casting, and temporal key derivation.

**This notebook contains no processing logic** — all transforms happen in `src/medallion/bronze.py`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from config import BRONZE_DIR, MEDALLION_METADATA_DIR

## Load Bronze manifest and data

In [ ]:
import json

manifest_path = MEDALLION_METADATA_DIR / "bronze_manifest.json"
if manifest_path.exists():
    with open(manifest_path) as f:
        manifest = json.load(f)
    print(f"Bronze manifest: {manifest['source_row_count']} rows ingested at {manifest['written_at']}")
    print(f"Columns: {len(manifest['columns'])}")
else:
    print("No Bronze manifest found. Run: make bronze")

In [ ]:
bronze_path = BRONZE_DIR / "transactions.parquet"
assert bronze_path.exists(), f"Bronze file not found: {bronze_path}"

df = pd.read_parquet(bronze_path)
print(f"Shape: {df.shape}")
df.dtypes

## Schema validation

In [ ]:
expected_dtypes = {
    "purchase_price": "float64",
    "sqm": "float64",
    "sqm_price": "float64",
    "year_build": "Int64",
    "no_rooms": "Int64",
}

for col, expected in expected_dtypes.items():
    actual = str(df[col].dtype)
    status = "OK" if actual == expected else f"MISMATCH ({actual})"
    print(f"  {col}: {status}")

assert pd.api.types.is_datetime64_any_dtype(df["date"]), "date should be datetime"
print("  date: OK (datetime)")

assert (df["zip_code"].str.len() == 4).all(), "zip_code should be 4 chars"
print("  zip_code: OK (zero-padded 4-digit)")

## Temporal keys

In [ ]:
assert df["quarter_id"].str.match(r"^\d{4}-Q[1-4]$").all(), "Invalid quarter_id format"

year_range = df["year_sale"].dropna()
print(f"Year range: {int(year_range.min())} - {int(year_range.max())}")
print(f"Quarters: {df['quarter_id'].nunique()} unique")

## Null counts

In [ ]:
nulls = df.isnull().sum()
nulls_pct = (nulls / len(df) * 100).round(2)
null_report = pd.DataFrame({"null_count": nulls, "null_pct": nulls_pct})
null_report[null_report["null_count"] > 0].sort_values("null_pct", ascending=False)

## Transaction volume over time

In [ ]:
quarterly_counts = df.groupby("quarter_id").size().sort_index()

fig, ax = plt.subplots(figsize=(14, 4))
quarterly_counts.plot(ax=ax, kind="bar", width=0.8, color="steelblue", alpha=0.7)
ax.set_title("Bronze: Transactions per Quarter")
ax.set_ylabel("Count")
ax.set_xlabel("")
ax.xaxis.set_major_locator(plt.MaxNLocator(20))
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()